# Char-level RNN Language Model

Train a character-level LSTM on **your own dataset** and compare three sampling strategies.

**What to do**
1. Load your dataset file (in Colab: upload it via the *Files* panel).
2. Edit the configuration cell below to fit your text. Explore the notebook and its results to find out how best to do this.
3. Run all cells.
4. Check that the three cells under **Submission outputs** show your generated text.

**Dataset size:** aim for medium sized texts (**50,000–300,000 characters**). Small texts get memorized, large texts train slowly. If it is too slow on your laptop, use Colab with a GPU runtime (*Runtime → Change runtime type*).

**Submit on Moodle:** this notebook **with all outputs**. Do not change the code in the *Submission outputs* section.

## 1. Configuration (edit only this cell)

In [ ]:
DATA_PATH  = "horoskope.csv"    # your dataset: .txt (plain UTF-8 text) or .csv
CSV_SEP    = "\t"               # only for .csv: column separator
CSV_COLUMN = "text"             # only for .csv: name of the column containing the text
LOWERCASE  = True               # convert text (and prompt) to lower case -> smaller vocabulary. Play around with this parameter and see how the outputs change

PROMPT     = "heute "           # start text for generation, should fit your dataset
GEN_LENGTH = 300                # number of characters to generate
EPOCHS     = 150                # training epochs. Use a low number here to start (~50)
SEED       = 42                 # random seed, keeps runs comparable

In [ ]:
import time

import pandas as pd
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (training runs on CPU)")

## 2. Loading the dataset

In [ ]:
def load_text(path):
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path, sep=CSV_SEP)
        text = df[CSV_COLUMN].astype(str).str.cat(sep=" ")
    else:
        with open(path, encoding="utf-8") as f:
            text = f.read()
    return text.lower() if LOWERCASE else text


text = load_text(DATA_PATH)
prompt = PROMPT.lower() if LOWERCASE else PROMPT

print(f"Dataset: {DATA_PATH}")
print(f"Length of text: {len(text):,} characters")

print("\nFirst 300 characters:\n")
print(text[:300])

## 3. Text Processing

In [ ]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text


def make_dataset(text, SEQ_LEN=100, BATCH_SIZE=64, BUFFER_SIZE=10000):
    vocab = sorted(set(text))
    print(f'{len(vocab)} unique characters')

    ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
    chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

    all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))  # text into single chars -> to char ids
    ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

    sequences = ids_dataset.batch(SEQ_LEN+1, drop_remainder=True)  # throw away last short chunk

    dataset = sequences.map(split_input_target)  # apply split_input_target on each chunk

    for input_example, target_example in dataset.take(1):
        print("Input :", tf.strings.reduce_join(chars_from_ids(input_example), axis=-1).numpy())
        print("Target:", tf.strings.reduce_join(chars_from_ids(target_example), axis=-1).numpy())

    n_batches = len(text) // (SEQ_LEN + 1) // BATCH_SIZE
    print(f"{n_batches} training batches per epoch")
    if n_batches == 0:
        raise ValueError(f"Dataset too small: need at least {(SEQ_LEN + 1) * BATCH_SIZE:,} characters.")

    dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.experimental.AUTOTUNE)

    return dataset, ids_from_chars, chars_from_ids, ids_from_chars.get_vocabulary()

In [ ]:
dataset, ids_from_chars, chars_from_ids, vocab = make_dataset(text)

unknown = sorted(set(prompt) - set(vocab))
if unknown:
    print(f"WARNING: the prompt contains characters that do not occur in the dataset: {unknown}")

## 4. Model Setup
This section defines the model as a `keras.Model` subclass (For details see [Making new Layers and Models via subclassing](https://www.tensorflow.org/guide/keras/custom_layers_and_models)).

This model has three layers:

* `tf.keras.layers.Embedding`: The input layer. A trainable lookup table that will map each character-ID to a vector with `embedding_dim` dimensions that, after training, are meaningful in representation.
* `tf.keras.layers.LSTM`: A type of RNN with size `units=rnn_units` (You can also use a GRU layer here.)
* `tf.keras.layers.Dense`: The output layer, with `vocab_size` outputs. It outputs one logit for each character in the vocabulary. Output: How likely is each character according to the model.

In [ ]:
class MyLSTM(tf.keras.Model):
    def __init__(self, vocab_size, EMBEDDING_DIM=256, RNN_UNITS=1024):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, EMBEDDING_DIM)
        self.lstm = tf.keras.layers.LSTM(RNN_UNITS, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = inputs
        x = self.embedding(x, training=training)
        if states is None:  # first call, no state yet
            batch_size = tf.shape(inputs)[0]
            states = self.lstm.get_initial_state(batch_size)
        x, state_h, state_c = self.lstm(x, training=training, initial_state=states) # Pass 'states' directly
        states = [state_h, state_c]
        x = self.dense(x, training=training)

        if return_state:
            return x, states
        else:
            return x

def build_model(vocab_size):
    model = MyLSTM(vocab_size)
    loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(loss=loss, optimizer='adam')
    return model

## 5. Training

In [ ]:
model = build_model(len(vocab))

start = time.time()
history = model.fit(dataset, epochs=EPOCHS, verbose=2)
print(f"\nTraining took {(time.time() - start) / 60:.1f} min, final loss: {history.history['loss'][-1]:.4f}")

## 6. Text Generation

The model predicts a distribution over the next character. How we *sample* from it changes the output:

* **Argmax:** always pick the most probable character (greedy decoding).
* **Random sampling:** sample from the full distribution, each character with its predicted probability.
* **Temperature sampling:** divide the logits by `t` before sampling. `t < 1` makes the output more conservative, `t > 1` more random.
* **Top-k sampling:** only sample from the `k` most likely characters.
* **Nucleus (top-p) sampling:** only sample from the smallest set of characters whose probabilities add up to at least `p`.

The following makes a single step prediction given a trained model:

In [ ]:
class OneStep(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars):
        super().__init__()
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # Create a mask to prevent "[UNK]" from being generated.
        skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
        sparse_mask = tf.SparseTensor(
            # Put a -inf at each bad index.
            values=[-float('inf')]*len(skip_ids),
            indices=skip_ids,
            # Match the shape to the vocabulary
            dense_shape=[len(ids_from_chars.get_vocabulary())])
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convert strings to token IDs.
        input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # Run the model.
        # predicted_logits.shape is [batch, char, next_char_logits]
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                            return_state=True)
        # Only use the last prediction.
        predicted_logits = predicted_logits[:, -1, :]
        # Apply the prediction mask: prevent "[UNK]" from being generated.
        predicted_logits = predicted_logits + self.prediction_mask

        # Sample the output logits to generate token IDs.
        predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)  # random sampling
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert from token ids to characters
        predicted_chars = self.chars_from_ids(predicted_ids)

        # Return the characters and model state.
        return predicted_chars, states

In [ ]:
class OneStepTemperature(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # Create a mask to prevent "[UNK]" from being generated.
        skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
        sparse_mask = tf.SparseTensor(
            # Put a -inf at each bad index.
            values=[-float('inf')]*len(skip_ids),
            indices=skip_ids,
            # Match the shape to the vocabulary
            dense_shape=[len(ids_from_chars.get_vocabulary())])
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convert strings to token IDs.
        input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # Run the model.
        # predicted_logits.shape is [batch, char, next_char_logits]
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                            return_state=True)
        # Only use the last prediction.
        predicted_logits = predicted_logits[:, -1, :]
        predicted_logits = predicted_logits/self.temperature
        # Apply the prediction mask: prevent "[UNK]" from being generated.
        predicted_logits = predicted_logits + self.prediction_mask

        # Sample the output logits to generate token IDs.
        predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert from token ids to characters
        predicted_chars = self.chars_from_ids(predicted_ids)

        # Return the characters and model state.
        return predicted_chars, states

In [ ]:
class OneStepTopK(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, topk=1):
        super().__init__()
        self.topk = topk
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # Create a mask to prevent "[UNK]" from being generated.
        skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
        sparse_mask = tf.SparseTensor(
            # Put a -inf at each bad index.
            values=[-float('inf')]*len(skip_ids),
            indices=skip_ids,
            # Match the shape to the vocabulary
            dense_shape=[len(ids_from_chars.get_vocabulary())])
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convert strings to token IDs.
        input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # Run the model.
        # predicted_logits.shape is [batch, char, next_char_logits]
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                            return_state=True)
        # Only use the last prediction.
        predicted_logits = predicted_logits[:, -1, :]
        # Apply the prediction mask: prevent "[UNK]" from being generated.
        predicted_logits = predicted_logits + self.prediction_mask
        # Select top k predictions
        top_k_logits, top_k_indices = tf.math.top_k(predicted_logits, k=self.topk, sorted=True)
        # Sample the output logits to generate token IDs.
        predicted_pos = tf.random.categorical(top_k_logits, num_samples=1, dtype=top_k_indices.dtype)
        predicted_ids = tf.gather(tf.squeeze(top_k_indices), predicted_pos)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert from token ids to characters
        predicted_chars = self.chars_from_ids(predicted_ids)

        # Return the characters and model state.
        return predicted_chars, states

In [ ]:
class OneStepTopP(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, topp=1.0):
        super().__init__()
        self.topp = topp
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # Create a mask to prevent "[UNK]" from being generated.
        skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
        sparse_mask = tf.SparseTensor(
            # Put a -inf at each bad index.
            values=[-float('inf')]*len(skip_ids),
            indices=skip_ids,
            # Match the shape to the vocabulary
            dense_shape=[len(ids_from_chars.get_vocabulary())])
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convert strings to token IDs.
        input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # Run the model.
        # predicted_logits.shape is [batch, char, next_char_logits]
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                            return_state=True)
        # Only use the last prediction.
        predicted_logits = predicted_logits[:, -1, :]
        # Apply the prediction mask: prevent "[UNK]" from being generated.
        predicted_logits = predicted_logits + self.prediction_mask
        # Select top p predictions: sort all logits, then sum up their probabilities
        top_k_logits, top_k_indices = tf.math.top_k(predicted_logits, k=predicted_logits.shape[-1])
        cumulative_probs = tf.math.cumsum(tf.nn.softmax(top_k_logits, axis=-1), axis=-1)
        score_mask = cumulative_probs < self.topp
        # Shift right by one: always keep the most likely character and the one that crosses p
        score_mask = tf.concat((tf.ones([score_mask.shape[0], 1], dtype=tf.bool), score_mask[:, :-1]), axis=-1)
        topp_next_scores = tf.where(score_mask, top_k_logits, tf.fill(top_k_logits.shape, -float("Inf")))
        # Sample the output logits to generate token IDs.
        predicted_pos = tf.random.categorical(topp_next_scores, num_samples=1, dtype=top_k_indices.dtype)
        predicted_ids = tf.gather(tf.squeeze(top_k_indices), predicted_pos)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert from token ids to characters
        predicted_chars = self.chars_from_ids(predicted_ids)

        # Return the characters and model state.
        return predicted_chars, states

In [ ]:
def generate(one_step_model, prompt=prompt, length=GEN_LENGTH, seed=SEED):
    tf.random.set_seed(seed)
    states = None
    next_char = tf.constant([prompt])
    result = [next_char]

    for _ in range(length):
        next_char, states = one_step_model.generate_one_step(next_char, states=states)
        result.append(next_char)

    return tf.strings.join(result)[0].numpy().decode('utf-8')


def show(title, one_step_model):
    print(f"{title}  |  prompt: {prompt!r}  |  seed: {SEED}\n")
    print(generate(one_step_model))

## 7. Submission outputs (do not change)

These three cells are graded. They use fixed sampling parameters so that all submissions are comparable.

In [ ]:
show("Temperature sampling (t = 0.8)", OneStepTemperature(model, chars_from_ids, ids_from_chars, temperature=0.8))

In [ ]:
show("Top-k sampling (k = 5)", OneStepTopK(model, chars_from_ids, ids_from_chars, topk=5))

In [ ]:
show("Nucleus sampling (p = 0.9)", OneStepTopP(model, chars_from_ids, ids_from_chars, topp=0.9))